In [ ]:
# DDGR20，嵌入近似hint，l-noisy < <v,s> < l+noisy
cd framework

In [ ]:
load("../framework/instance_gen.sage")
import numpy as np
import random

In [ ]:
n = 768
m = n
q = 3329
D_s = build_centered_binomial_law(2)
D_e = D_s
A, b, s, dbdd = initialize_from_LWE_instance(DBDD_predict, n, q, m, D_e, D_s)
# _ = dbdd.integrate_q_vectors(q, report_every=20)
beta, delta = dbdd.estimate_attack()

In [ ]:
def generate_se_eta_bias_approx_hint(m, n, bias, k):
    V = []
    L = []

    for i in range(k):
        # D_e = {-3: 1/64, -2: 6/64, -1: 15/64, 0: 20/64, 1: 15/64, 2: 6/64, 3: 1/64}
        D_e = {-2: 1/16, -1: 4/16, 0: 6/16, 1: 4/16, 2: 1/16}
        values, probabilities = zip(*D_e.items())
        v = np.array(np.random.choice(values, size=m+n, p=probabilities))
        noisy = random.randint(-bias, bias)
        l = dbdd.leak(v)+noisy
        V.append(v)
        L.append(l)
    print("L",L)
    return V,L

In [ ]:
nph_Kyber512 = [0, 40, 80, 120, 160, 200, 240, 280, 320, 360, 400, 440, 480, 520, 560, 600, 640, 680, 720, 760, 800, 840, 880, 920, 960, 1000, 1500, 2000, 2500, 3000, 3500, 4000]
dis_Kyber512_our = [39.56, 40.03, 40.4, 40.4, 40.31, 40.09, 39.85, 39.7, 39.23, 39.2, 38.86, 38.64, 38.33, 37.92, 37.5, 37.2, 37.1, 36.7, 36.54, 36.23, 35.97, 35.76, 35.48, 35.16, 34.96, 34.77, 31.73, 28.8, 26.84, 24.98, 23.22, 21.74]
nph_Kyber768 = [0,40,80,120,160,200,240,280,320,360,400,440,480,520,560,600,640,680,720,760,800,840,880,920,960,1000,1040,1080,1120,1160,1200,1240,1280,1320,1360,1400,1440,1480,1520,1560,1600,1640,1680,1720,1760,1800,1840,1880,1920,1960,2000]
dis_Kyber768_our = [38.48, 38.6, 39.23, 39.48, 39.14, 38.97, 38.81, 38.85, 38.45, 38.41, 38.06, 37.85, 37.73, 37.05, 36.81, 36.46, 36.2, 36.07, 35.88, 35.33, 35.22, 35.08, 34.74, 34.12, 34.15, 33.76, 33.71, 33.56, 33.35, 32.97, 32.68, 32.4, 32.39, 32.26, 32.09, 31.45, 31.38, 31.05, 30.72, 30.62, 30.44, 30.44, 30.02, 29.6, 29.4, 29.61, 29.45, 28.8, 29.03, 28.89, 28.19]
nph_Kyber1024 = [0,40,80,120,160,200,240,280,320,360,400,440,480,520,560,600,640,680,720,760,800,840,880,920,960,1000,1040,1080,1120,1160,1200,1240,1280,1320,1360,1400,1440,1480,1520,1560,1600,1640,1680,1720,1760,1800,1840,1880,1920,1960,2000,2500,3000,3500,4000,4500,5000,5500,6000,6500,7000,7500,8000]
dis_Kyber1024_our = [45.4,45.41,45.41,45.42,45.37,45.29,45.17,44.97,44.78,44.55,44.35,44,43.82,43.64,43.26,43.13,42.86,42.55,42.21,42.09,41.77,41.55,41.32,41.01,40.81,40.51,40.15,40.12,39.76,39.56,39.31,39.17,38.9,38.58,38.43,38.16,37.93,37.75,37.54,37.33,37.04,36.92,36.7,36.48,36.21,35.93,35.9,35.67,35.48,35.28,34.98,33.06,31.19,29.8,28.57,27.46,26.4,25.35,24.64,23.57,23.05,22.31,21.45]

num_hint = 2001
bias = int(q/64)
V, L = generate_se_eta_bias_approx_hint(m, n, bias, num_hint)
BETA_ori = []
BETA_com = []
index = 0
for j in range(num_hint):
    if j == nph_Kyber768[index]:
        # _ = dbdd.integrate_q_vectors(q, report_every=20)
        beta_ori, delta = dbdd.estimate_attack()
        print("beta_ori: ", beta_ori)
        BETA_ori.append(beta_ori)
        st = dis_Kyber768_our[index]/dis_Kyber768_our[0]
        beta_com, delta = dbdd.estimate_attack_SMY(st)
        print("beta_com: ", beta_com)
        BETA_com.append(beta_com)
        index += 1
    print("the ",j+1,"-th secret error sca approx hint")
    _ = dbdd.integrate_approx_hint(vec(V[j]), L[j], bias, aposteriori=False)
print("BETA_ori",BETA_ori)
print("BETA_com",BETA_com)